# waveletcnn on the FPGA

Cloud / land / sea classification of a HYPSO-2 capture on the FPGA. The capture is cut
into 32 x 32 pixel patches; for each one the notebook drops 10 bands and z-scores the
rest as in training, the kernel runs the quantized WaveletCNN and returns 3 scores per
pixel. The highest score is the class: **0 cloud, 1 land, 2 sea**.

Next to this notebook: `waveletcnn.bit` + `.hwh` (the design),
`xwaveletcnn_hw.h` (register offsets), `mu_sd.txt` (training mean/std per band), and
an image zip made on the PC:

```bash
python3 tools/waveletcnn_image.py prepare CAPTURE-l1a.nc --labels CAPTURE-l1a_labels.npy     --crop 200 300 128 256 --out aegean_waveletcnn
source fpga.sh; fpga_zip aegean_waveletcnn
```

Run the cells from the top.

The image zip can also go in the folder above this one, once for every design: the notebook looks there when it is not next to it, unpacks it there, and writes its own results next to itself.

## 1. Load the bitstream and give the kernel its two DDR buffers

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

overlay = Overlay('waveletcnn.bit')         # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                     # the HLS block, named 'kernel' in the block design

# Register offsets from the HLS driver header.
header = Path('xwaveletcnn_hw.h').read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, DIN, DOUT = register('AP_CTRL'), register('DIN_DATA'), register('DOUT_DATA')

# The kernel reads one patch from patch_in and writes its scores to patch_out,
# at these physical addresses (64 bits: low word, then high word).
patch_in = allocate(shape=(32, 32, 110), dtype=np.float32)
patch_out = allocate(shape=(32, 32, 3), dtype=np.float32)
for offset, buffer in ((DIN, patch_in), (DOUT, patch_out)):
    kernel.write(offset, buffer.physical_address & 0xFFFFFFFF)
    kernel.write(offset + 4, buffer.physical_address >> 32)
print('kernel idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. The functions: preprocess, one patch, a whole image

In [ ]:
# Training's preprocessing: keep 110 of the 120 bands, then (x - mean) / std.
mu, sd = [np.array(a.replace(',', ' ').split(), dtype=np.float64)
          for a in re.findall(r'\[([^\]]*)\]', Path('mu_sd.txt').read_text())]
KEPT = [b for b in range(120) if b not in (0, 1, 2, 3, 4, 5, 6, 7, 118, 119)]
MEAN, INV_STD = mu.astype(np.float32), (1 / sd).astype(np.float32)

def preprocess(raw):
    return (raw[..., KEPT].astype(np.float32) - MEAN) * INV_STD

def run_patch(x, timeout=60):
    """One preprocessed 32 x 32 x 110 patch on the FPGA -> 32 x 32 x 3 scores."""
    patch_in[:] = x
    patch_in.flush()                       # into DDR, not just the CPU cache
    kernel.write(AP_CTRL, 1)               # start
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:  # wait for done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')
    patch_out.invalidate()                 # read what the kernel wrote
    return np.array(patch_out)

def pad(cube):
    """Grow the image to whole 32 x 32 patches by repeating its last row and column."""
    h, w, _ = cube.shape
    return np.pad(cube, ((0, -h % 32), (0, -w % 32), (0, 0)), mode='edge')

def classify(cube):
    """Raw cube (H x W x 120) -> scores (H x W x 3)."""
    h, w, _ = cube.shape
    padded = pad(cube)
    scores = np.zeros(padded.shape[:2] + (3,), dtype=np.float32)
    start = time.monotonic()
    for y in range(0, padded.shape[0], 32):
        for x in range(0, padded.shape[1], 32):
            scores[y:y + 32, x:x + 32] = run_patch(preprocess(padded[y:y + 32, x:x + 32]))
        print(f'  rows {y}-{y + 31} done, {time.monotonic() - start:.1f} s', flush=True)
    return scores[:h, :w]

## 3. Load the image

In [ ]:
# The image: IMAGE.zip or the folder IMAGE/, next to this notebook or in the folder above
# it. Above is shared by every design's folder: upload the image there once. A zip is
# unpacked next to itself, and again only when it is newer than what was unpacked.
IMAGE = 'aegean_waveletcnn'
place = next((d for d in (Path('.'), Path('..'))
              if (d / f'{IMAGE}.zip').exists() or (d / IMAGE / 'cube.npy').exists()), None)
if place is None:
    raise FileNotFoundError(f'no {IMAGE}.zip or {IMAGE}/ in {Path(".").resolve()} or the folder above it')
folder, archive = place / IMAGE, place / f'{IMAGE}.zip'
if archive.exists() and (not (folder / 'cube.npy').exists()
                         or (folder / 'cube.npy').stat().st_mtime < archive.stat().st_mtime):
    with zipfile.ZipFile(archive) as z:
        z.extractall(place)
print(f'image: {folder.resolve()}')
cube = np.load(folder / 'cube.npy')
# The numpy model's scores, made on the PC (waveletcnn_image.py prepare), if the image has them.
reference = np.load(folder / 'reference_scores.npy') if (folder / 'reference_scores.npy').exists() else None
if reference is None:
    print(f'no reference_scores.npy in {folder}: checked against the labels only')
meta = json.loads((folder / 'meta.json').read_text())
print(f'{cube.shape[0]} x {cube.shape[1]} pixels, {-(-cube.shape[0] // 32) * -(-cube.shape[1] // 32)} patches')

## 4. Self-test: the first patch against the PC's answer

In [ ]:
first = run_patch(preprocess(pad(cube)[:32, :32]))
if reference is not None:
    want = reference[:32, :32]
    error = np.abs(first[:want.shape[0], :want.shape[1]] - want).max()
    print('PASS' if error < 1e-3 * (1 + np.abs(want).max()) else 'FAIL', f'worst error {error:.2e}')
else:
    print('no reference: the first patch ran, its answers are not checked')

## 5. The whole image

In [ ]:
start = time.monotonic()
scores = classify(cube)
seconds = time.monotonic() - start
print(f'{seconds:.1f} s for {cube.shape[0]} x {cube.shape[1]} pixels')
np.save(f'{IMAGE}_fpga_scores.npy', scores)    # next to this notebook: the image folder is shared

## 6. Check it: against the labels, and the PC's answer if the image has it

In [ ]:
classes, close = scores.argmax(-1), None
if reference is not None:
    close = np.abs(scores - reference) <= 1e-3 + 1e-3 * np.abs(reference)
    print(f'scores within tolerance of the PC: {close.all(-1).mean():.2%}, '
          f'same class: {(classes == reference.argmax(-1)).mean():.2%}')
labels = np.load(folder / 'labels.npy') if (folder / 'labels.npy').exists() else None
if labels is not None:
    known = labels < 3
    print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%}')
    for c, name in enumerate(('cloud', 'land', 'sea')):
        union = ((classes == c) | (labels == c))[known].sum()
        iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
        print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = cube[:, :, meta['rgb_bands']].astype(np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          'FPGA classes': COLORS[classes]}
if labels is not None:
    panels['labels'] = COLORS[np.minimum(labels, 3)]
if close is not None:
    panels['differs from PC'] = ~close.all(-1)
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img, cmap='gray_r' if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # cloud white, land green, sea blue, gray = unlabeled